# Experiment 0

**Reasoning**:
Define the function `generate_sequence_pair` to generate random integer sequences and their reversed counterparts based on the provided instructions.



In [ ]:
import numpy as np

def generate_sequence_pair(seq_length, max_int):
  """Generates a random integer sequence and its reversed sequence.

  Args:
    seq_length: The desired length of the sequence.
    max_int: The maximum integer value in the sequence (inclusive).

  Returns:
    A tuple containing the original sequence and the reversed sequence.
  """
  original_sequence = np.random.randint(0, max_int + 1, size=seq_length)
  reversed_sequence = original_sequence[::-1]
  return original_sequence, reversed_sequence

# Example usage (for testing purposes)
# seq, reversed_seq = generate_sequence_pair(10, 100)
# print("Original sequence:", seq)
# print("Reversed sequence:", reversed_seq)

**Reasoning**:
Implement the `SequenceReverser` class with the specified layers and forward pass.



In [ ]:
import torch.nn as nn

class SequenceReverser(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(SequenceReverser, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.linear = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        # We only need the output from the last time step for a many-to-one scenario
        # For a many-to-many scenario (like sequence reversal), we need the output at each time step.
        # The output of LSTM is (batch_size, sequence_length, hidden_size)
        # We want to predict an output for each element in the input sequence.
        output = self.linear(lstm_out)
        return output

**Reasoning**:
Define the hyperparameters, instantiate the model, define the loss function and the optimizer as instructed.



In [ ]:
import torch
import torch.optim as optim

# 1. Define hyperparameters
vocab_size = 101  # Max integer value + 1 (0-100)
embedding_dim = 64
hidden_dim = 128
output_dim = vocab_size  # Output is the probability distribution over possible integers
learning_rate = 0.001
num_epochs = 10000
batch_size = 32
seq_length = 10 # Example sequence length

# 2. Instantiate the SequenceReverser model
model = SequenceReverser(vocab_size, embedding_dim, hidden_dim, output_dim)

# 3. Define the loss function
criterion = nn.CrossEntropyLoss()

# 4. Define the optimizer
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 5. Initialize a list to store training loss
training_loss_history = []

**Reasoning**:
Implement the training loop as described in the instructions, including data generation, forward pass, loss calculation, and backpropagation.



In [ ]:
for epoch in range(num_epochs):
    epoch_loss = 0
    correct_predictions = 0
    total_predictions = 0
    for _ in range(batch_size):
        original_sequences = []
        reversed_sequences = []
        original_sequence, reversed_sequence = generate_sequence_pair(seq_length, vocab_size - 1)
        original_sequences.append(original_sequence)
        reversed_sequences.append(reversed_sequence)

        original_sequences_tensor = torch.LongTensor(original_sequences)
        reversed_sequences_tensor = torch.LongTensor(reversed_sequences)

        optimizer.zero_grad()
        outputs = model(original_sequences_tensor)

        # Reshape outputs and targets for CrossEntropyLoss
        # outputs shape: (batch_size, seq_length, output_dim)
        # targets shape: (batch_size, seq_length)
        # CrossEntropyLoss expects input: (N, C, d1, d2, ...) and target: (N, d1, d2, ...)
        # where C is the number of classes.
        # So we need to permute the output to be (batch_size, output_dim, seq_length)
        outputs = outputs.permute(0, 2, 1)

        loss = criterion(outputs, reversed_sequences_tensor)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        total_predictions += reversed_sequences_tensor.size(1)
        correct_predictions += (predicted == reversed_sequences_tensor).sum().item()

    avg_epoch_loss = epoch_loss / batch_size
    epoch_accuracy = 100 * correct_predictions / total_predictions
    training_loss_history.append(avg_epoch_loss)

    if (epoch + 1) % 1 == 0: # Print loss and accuracy every epoch
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%')

Streaming output truncated to the last 5000 lines.
Epoch [5001/10000], Loss: 2.3130, Accuracy: 50.31%
Epoch [5002/10000], Loss: 2.3161, Accuracy: 50.62%
Epoch [5003/10000], Loss: 2.3083, Accuracy: 50.94%
Epoch [5004/10000], Loss: 2.3123, Accuracy: 50.62%
Epoch [5005/10000], Loss: 2.3032, Accuracy: 51.25%
Epoch [5006/10000], Loss: 2.3219, Accuracy: 50.31%
Epoch [5007/10000], Loss: 2.3240, Accuracy: 50.62%
Epoch [5008/10000], Loss: 2.3193, Accuracy: 50.62%
Epoch [5009/10000], Loss: 2.3173, Accuracy: 50.00%
Epoch [5010/10000], Loss: 2.3164, Accuracy: 50.94%
Epoch [5011/10000], Loss: 2.3235, Accuracy: 50.31%
Epoch [5012/10000], Loss: 2.3119, Accuracy: 50.00%
Epoch [5013/10000], Loss: 2.3187, Accuracy: 50.31%
Epoch [5014/10000], Loss: 2.3147, Accuracy: 51.25%
Epoch [5015/10000], Loss: 2.3112, Accuracy: 50.31%
Epoch [5016/10000], Loss: 2.3068, Accuracy: 50.62%
Epoch [5017/10000], Loss: 2.3145, Accuracy: 50.62%
Epoch [5018/10000], Loss: 2.3209, Accuracy: 50.31%
Epoch [5019/10000], Loss: 2.318

**Reasoning**:
Implement the evaluation steps as described in the instructions, including generating a new sequence pair, converting to tensor, setting the model to evaluation mode, disabling gradient calculation, getting model predictions, extracting the predicted sequence, and setting the model back to training mode.



In [ ]:
# 1. Generate a new, unseen original sequence and its corresponding reversed sequence
original_sequence_eval, reversed_sequence_eval = generate_sequence_pair(seq_length, vocab_size - 1)
print("Original sequence (evaluation):", original_sequence_eval)
print("Reversed sequence (evaluation):", reversed_sequence_eval)

# 2. Convert the original sequence to a PyTorch LongTensor and add a batch dimension
original_sequence_eval_tensor = torch.LongTensor([original_sequence_eval])

# 3. Set the model to evaluation mode
model.eval()

# 4. Use torch.no_grad() to disable gradient calculation
with torch.no_grad():
    # 5. Pass the original sequence tensor through the trained model to get the predicted output
    output_eval = model(original_sequence_eval_tensor)

    # 6. Get the predicted integer sequence
    # The output_eval has shape (batch_size, seq_length, output_dim)
    # We need to find the index with the maximum value along the output_dim for each step
    predicted_sequence_eval = torch.argmax(output_eval, dim=2).squeeze().numpy()

print("Predicted sequence (evaluation):", predicted_sequence_eval)

# 7. Set the model back to training mode
model.train()

Original sequence (evaluation): [37 71 57 91 27 15 31 72 87 87]
Reversed sequence (evaluation): [87 87 72 31 15 27 91 57 71 37]
Predicted sequence (evaluation): [60 60  2 71 77 27 91 57 71 37]


SequenceReverser(
  (embedding): Embedding(101, 64)
  (lstm): LSTM(64, 128, batch_first=True)
  (linear): Linear(in_features=128, out_features=101, bias=True)
)

**Reasoning**:
Compare the predicted and expected sequences and print the result.



In [ ]:
# Compare the predicted sequence with the expected reversed sequence
sequences_match = np.array_equal(predicted_sequence_eval, reversed_sequence_eval)

# Print the result
if sequences_match:
    print("Model successfully reversed the sequence.")
else:
    print("Model failed to reverse the sequence.")
    print("Predicted sequence:", predicted_sequence_eval)
    print("Expected sequence:", reversed_sequence_eval)

Model failed to reverse the sequence.
Predicted sequence: [58  9  9 50 57 57  9 49 99 96]
Expected sequence: [89 27 38 33 17 77  5 18 46 58]


# Experiment 1

In [ ]:
import string
import re
from numpy import array, argmax, random, take
import pandas as pd
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding, RepeatVector
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.callbacks import ModelCheckpoint
from keras.preprocessing.sequence import pad_sequences
from keras.models import load_model
from keras import optimizers
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option('display.max_colwidth', 200)

In [ ]:
data_path = '/content/mar.txt'
with open(data_path, 'r', encoding='utf-8') as f:
    lines = f.read()
lines

'Go.\tजा.\tCC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #3138228 (sabretou)\nRun!\tपळ!\tCC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #3138217 (sabretou)\nRun!\tधाव!\tCC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #3138218 (sabretou)\nRun!\tपळा!\tCC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #3138219 (sabretou)\nRun!\tधावा!\tCC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #3138220 (sabretou)\nWho?\tकोण?\tCC-BY 2.0 (France) Attribution: tatoeba.org #2083030 (CK) & #3138225 (sabretou)\nWow!\tवाह!\tCC-BY 2.0 (France) Attribution: tatoeba.org #52027 (Zifre) & #6728118 (sabretou)\nDuck!\tखाली वाका!\tCC-BY 2.0 (France) Attribution: tatoeba.org #280158 (CM) & #7731217 (Nativemarathi)\nFire!\tआग!\tCC-BY 2.0 (France) Attribution: tatoeba.org #1829639 (Spamster) & #3232248 (sabretou)\nFire!\tफायर!\tCC-BY 2.0 (France) Attribution: tatoeba.org #1829639 (Spamster) & #3232249 (sabretou)\nHelp!\tवाचवा!\tCC-BY 2.0 

In [ ]:
# Split a text into sentences
def to_lines(text):
	sents = text.strip().split('\n')
	sents = [i.split('\t') for i in sents]
	return sents

In [ ]:
eng_mar = to_lines(lines)
eng_mar[:5]

[['Go.',
  'जा.',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #3138228 (sabretou)'],
 ['Run!',
  'पळ!',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #3138217 (sabretou)'],
 ['Run!',
  'धाव!',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #3138218 (sabretou)'],
 ['Run!',
  'पळा!',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #3138219 (sabretou)'],
 ['Run!',
  'धावा!',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #3138220 (sabretou)']]

In [ ]:
eng_mar = array(eng_mar)
eng_mar[:5]

array([['Go.', 'जा.',
        'CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #3138228 (sabretou)'],
       ['Run!', 'पळ!',
        'CC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #3138217 (sabretou)'],
       ['Run!', 'धाव!',
        'CC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #3138218 (sabretou)'],
       ['Run!', 'पळा!',
        'CC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #3138219 (sabretou)'],
       ['Run!', 'धावा!',
        'CC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #3138220 (sabretou)']],
      dtype='<U194')

In [ ]:
eng_mar.shape

(48389, 3)

In [ ]:
eng_mar = eng_mar[:30000,:]

In [ ]:
eng_mar = eng_mar[:,[0,1]]
eng_mar[:5]

array([['Go.', 'जा.'],
       ['Run!', 'पळ!'],
       ['Run!', 'धाव!'],
       ['Run!', 'पळा!'],
       ['Run!', 'धावा!']], dtype='<U194')

**Data Cleaning**

In [ ]:
# Remove Punctuation
eng_mar[:,0] = [s.translate(str.maketrans('', '', string.punctuation)) for s in eng_mar[:,0]]
eng_mar[:,1] = [s.translate(str.maketrans('', '', string.punctuation)) for s in eng_mar[:,1]]
eng_mar[:5]

array([['Go', 'जा'],
       ['Run', 'पळ'],
       ['Run', 'धाव'],
       ['Run', 'पळा'],
       ['Run', 'धावा']], dtype='<U194')

In [ ]:
# Convert text to lowercase
for i in range(len(eng_mar)):
	eng_mar[i,0] = eng_mar[i,0].lower()
eng_mar

array([['go', 'जा'],
       ['run', 'पळ'],
       ['run', 'धाव'],
       ...,
       ['the book fell to the floor', 'पुस्तक जमिनीवर पडलं'],
       ['the books are on the table', 'पुस्तकं टेबलावर आहेत'],
       ['the bride suddenly laughed', 'नवरी अचानक हसली']], dtype='<U194')

**Text to Sequence Conversion (word to index mapping)**

1. Convert Sentences into numbers.
2. Every Sentence should be of same length.

In [ ]:
# Function to build a tokenizer
def tokenization(lines):
  tokenizer = Tokenizer()
  tokenizer.fit_on_texts(lines)
  return tokenizer

In [ ]:
# Prepare English tokenizer
eng_tokenizer = tokenization(eng_mar[:, 0])
eng_vocab_size = len(eng_tokenizer.word_index) + 1

eng_length = 8
print('English Vocabulary Size: %d' % eng_vocab_size)

English Vocabulary Size: 3836


In [ ]:
# Prepare Marathi Tokenizer
mar_tokenizer = tokenization(eng_mar[:, 1])
mar_vocab_size = len(mar_tokenizer.word_index) + 1

mar_length = 8
print('Marathi Vocabulary Size: %d' % mar_vocab_size)

Marathi Vocabulary Size: 8466


In [ ]:
# Encode and Pad Sequences
def encode_sequences(tokenizer, length, lines):
	# Integer encode Sequences
	seq = tokenizer.texts_to_sequences(lines)
	# Pad Sequences with 0 values
	seq = pad_sequences(seq, maxlen=length, padding='post')
	return seq

It's time to encode the sentences. We will encode English Sentences as the input sequences and print Marathi Sequences as the target sequences. This has to be done for both the train and the test datasets.

In [ ]:
from sklearn.model_selection import train_test_split

# Split data into train and test sets
train, test = train_test_split(eng_mar, test_size=0.2, random_state = 12)

In [ ]:
# Prepare training data
trainX = encode_sequences(eng_tokenizer, eng_length, train[:, 0])
trainY = encode_sequences(mar_tokenizer, mar_length, train[:, 1])

In [ ]:
# Prepare validation data
testX = encode_sequences(eng_tokenizer, eng_length, test[:, 0])
testY = encode_sequences(mar_tokenizer, mar_length, test[:, 1])

**Define our Seq2Seq model architecture**

In [ ]:
# Build NMT model
def define_model(in_vocab,out_vocab, in_timesteps,out_timesteps,units):
  model = Sequential()
  model.add(Embedding(in_vocab, units, input_length=in_timesteps, mask_zero=True))
  model.add(LSTM(units))
  model.add(RepeatVector(out_timesteps))
  model.add(LSTM(units, return_sequences=True))
  model.add(Dense(out_vocab, activation='softmax'))
  return model

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# We use RMSprop optimizer because it is good choice when working with Recurrent Neural Networks
# Model Compilation
model = define_model(eng_vocab_size, mar_vocab_size, eng_length, mar_length, 192)
rms = optimizers.RMSprop(learning_rate=0.001)
model.compile(optimizer=rms, loss='sparse_categorical_crossentropy')

In [ ]:
# Let's train model now
# Train Model
history = model.fit(trainX, trainY.reshape(trainY.shape[0], trainY.shape[1], 1),
                    epochs=100, batch_size=128,
                    validation_split = 0.2)

Epoch 1/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 5s 35ms/step - loss: 1.0188 - val_loss: 1.7516
Epoch 2/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - loss: 0.9973 - val_loss: 1.7455
Epoch 3/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 6s 37ms/step - loss: 0.9899 - val_loss: 1.7367
Epoch 4/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 5s 35ms/step - loss: 0.9798 - val_loss: 1.7377
Epoch 5/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step - loss: 0.9599 - val_loss: 1.7323
Epoch 6/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 10s 36ms/step - loss: 0.9592 - val_loss: 1.7297
Epoch 7/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 10s 36ms/step - loss: 0.9472 - val_loss: 1.7224
Epoch 8/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - loss: 0.9332 - val_loss: 1.7231
Epoch 9/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 10s 36ms/step - loss: 0.9194 - val_loss: 1.7167
Epoch 10/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 10s 36ms/step - loss: 0.9073 - val_loss: 1.7253
Epoch 11/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - loss: 0.8996 - val_loss: 1.7092
Epoch 12/100
150/150 ━━

In [ ]:
import numpy as np

In [ ]:
# Prediction (important: keep batch_size small to avoid GPU OOM)
preds_proba = model.predict(testX, batch_size=128)
preds = np.argmax(preds_proba, axis=-1)
print(preds)

47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step
[[   8    3    6 ...    0    0    0]
 [  84  274  143 ...    0    0    0]
 [   4   65 1877 ...    0    0    0]
 ...
 [ 458  159  159 ...    0    0    0]
 [ 138  188   57 ...    0    0    0]
 [   4 2109  641 ...    0    0    0]]


In [ ]:
preds = array(preds)
preds

array([[   8,    3,    6, ...,    0,    0,    0],
       [  84,  274,  143, ...,    0,    0,    0],
       [   4,   65, 1877, ...,    0,    0,    0],
       ...,
       [ 458,  159,  159, ...,    0,    0,    0],
       [ 138,  188,   57, ...,    0,    0,    0],
       [   4, 2109,  641, ...,    0,    0,    0]])

In [ ]:
# Convert sequence of integers into corresponding words
def get_word(n, tokenizer):
  for word, index in tokenizer.word_index.items():
    if index == n:
      return word
  return None

In [ ]:
# Convert predictions into sentences (Marathi)
preds_text = []
for i in preds:
  temp = []
  for j in range(len(i)):
    t = get_word(i[j], mar_tokenizer)
    if j > 0:
      if (t == get_word(i[j-1], mar_tokenizer)) or (t == None):
        temp.append('')
      else:
        temp.append(t)
    else:
      if (t == None):
        temp.append('')
      else:
        temp.append(t)
  preds_text.append(' '.join(temp))

**Let's put original Marathi Sentences and the predeicted Marathi Senetences in a dataframe**

In [ ]:
pred_df = pd.DataFrame({'english': test[:, 0], 'actual' : test[:,1], 'predicted' : preds_text})

In [ ]:
# Print 15 rows randomly
pred_df.sample(15)

,english,actual,predicted
5939,who killed tom,टॉमला कोणी ठार मारलं,टॉमला कोणी सांगितलं
3380,is this made here,हे इथे बनवलं जातं का,हा इथे निघणार आहे का
3846,i feel like crying now,मला आता रडावसं वाटतंय,मला आत्ता बरं वाटतंय
3451,they can do anything,ते काहीही करू शकतात,त्या काहीही करू शकतात
2203,everyone got sick,सगळेच आजारी पडले,सगळेच आजारी बोलतात
5372,im going to try it,मी करून बघणार आहे,मी काहीही करणार करतोय
1832,tom speaks french too,टॉम फ्रेंचसुद्धा बोलतो,टॉम फ्रेंच बोलतो
4328,i talked,मी बोललो,मी बोलले
563,everybody sang,सर्वजण गायले,सगळे गायले
812,whos going to drive,चालवणार कोण आहे,मॅच थंड आहे


# Experiment 2

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ==============================================================================
# Assignment Part 1: Self-Attention
# ==============================================================================
class SelfAttention(nn.Module):
    """A from-scratch implementation of a single Self-Attention head."""
    def __init__(self, embed_size):
        super(SelfAttention, self).__init__()
        self.embed_size = embed_size
        self.queries = nn.Linear(embed_size, embed_size, bias=False)
        self.keys = nn.Linear(embed_size, embed_size, bias=False)
        self.values = nn.Linear(embed_size, embed_size, bias=False)

    def forward(self, x):
        Q = self.queries(x)
        K = self.keys(x)
        V = self.values(x)

        attention_scores = torch.matmul(Q, K.transpose(-2, -1))
        d_k = K.size(-1)
        scaled_scores = attention_scores / math.sqrt(d_k)

        attention_weights = F.softmax(scaled_scores, dim=-1)
        context_vectors = torch.matmul(attention_weights, V)

        return context_vectors, attention_weights

In [ ]:
# ==============================================================================
# Assignment Part 2: Multi-headed and Masked Self-Attention
# ==============================================================================
class MultiHeadAttention(nn.Module):
    """Implements Multi-Headed and Masked Self-Attention."""
    def __init__(self, embed_size, heads):
        super(MultiHeadAttention, self).__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        assert (
            self.head_dim * heads == embed_size
        ), "Embedding size must be divisible by the number of heads"

        self.qkv_layer = nn.Linear(embed_size, 3 * embed_size)
        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, x, mask=None):
        N, seq_length, _ = x.shape

        qkv = self.qkv_layer(x)
        qkv = qkv.reshape(N, seq_length, self.heads, 3 * self.head_dim)
        qkv = qkv.permute(0, 2, 1, 3)
        Q, K, V = qkv.chunk(3, dim=-1)

        energy = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e9"))

        attention = torch.softmax(energy, dim=-1)
        out = torch.matmul(attention, V)

        out = out.permute(0, 2, 1, 3).contiguous()
        out = out.reshape(N, seq_length, self.embed_size)

        return self.fc_out(out)

In [ ]:
# ==============================================================================
# Demonstration Block
# ==============================================================================
if __name__ == "__main__":
    # --- Configuration ---
    batch_size = 1
    sequence_length = 5
    embedding_size = 128
    num_heads = 4

    x_input = torch.rand(batch_size, sequence_length, embedding_size)

    # --- Part 1: Testing SelfAttention ---
    print("="*45)
    print("🚀 Running Assignment Part 1: Self-Attention")
    print("="*45)

    self_attention_layer = SelfAttention(embedding_size)
    output, weights = self_attention_layer(x_input)

    print(f"Input shape:  {x_input.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Weights shape: {weights.shape}")
    print("✅ Self-Attention implemented successfully.\n")

    # --- Part 2: Testing Multi-headed and Masked Self-Attention ---
    print("="*45)
    print("🚀 Running Assignment Part 2: Multi-Head & Masked")
    print("="*45)

    multi_head_layer = MultiHeadAttention(embedding_size, num_heads)

    # 2a. Multi-Head Attention (unmasked)
    multi_head_output = multi_head_layer(x_input)
    print("--- Multi-Head Attention (unmasked) ---")
    print(f"Input shape:  {x_input.shape}")
    print(f"Output shape: {multi_head_output.shape}\n")

    # 2b. Masked Multi-Head Attention
    causal_mask = torch.tril(torch.ones(sequence_length, sequence_length))
    masked_output = multi_head_layer(x_input, mask=causal_mask)
    print("--- Masked Multi-Head Attention ---")
    print(f"Output shape: {masked_output.shape}")
    print("="*45)

🚀 Running Assignment Part 1: Self-Attention
Input shape:  torch.Size([1, 5, 128])
Output shape: torch.Size([1, 5, 128])
Weights shape: torch.Size([1, 5, 5])
✅ Self-Attention implemented successfully.

🚀 Running Assignment Part 2: Multi-Head & Masked
--- Multi-Head Attention (unmasked) ---
Input shape:  torch.Size([1, 5, 128])
Output shape: torch.Size([1, 5, 128])

--- Masked Multi-Head Attention ---
Output shape: torch.Size([1, 5, 128])


# Experiment 3

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ---------------------------------------------------
# Step 1: Install Dependencies
# ---------------------------------------------------
!pip install datasets transformers sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 8.5 MB/s eta 0:00:00


In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00


In [ ]:
# ---------------------------------------------------
# Step 2: Import Libraries
# ---------------------------------------------------
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForSeq2Seq, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
import evaluate
import numpy as np

In [ ]:
# ---------------------------------------------------
# Step 3: Load Dataset (IITB English-Hindi)
# ---------------------------------------------------
dataset = load_dataset("cfilt/iitb-english-hindi")

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/190M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/85.7k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/500k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

In [ ]:
# Preview dataset
print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 1659083
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 520
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 2507
    })
})
{'translation': {'en': 'Give your application an accessibility workout', 'hi': 'अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें'}}


In [ ]:
# ---------------------------------------------------
# Step 4: Choose Pretrained Transformer Model
# ---------------------------------------------------
# MarianMT is a family of pretrained NMT models by Helsinki-NLP
model_checkpoint = "Helsinki-NLP/opus-mt-en-hi"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

In [ ]:
# ---------------------------------------------------
# Step 5: Preprocess Dataset
# ---------------------------------------------------
# We'll translate from English ("en") to Hindi ("hi")
source_lang = "en"
target_lang = "hi"
max_length = 128

In [ ]:
def preprocess_function(examples):
    inputs = [ex[source_lang] for ex in examples["translation"]]
    targets = [ex[target_lang] for ex in examples["translation"]]
    model_inputs = tokenizer(inputs, max_length=max_length, truncation=True)

    # Setup the tokenizer for targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=max_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
# Apply preprocessing
tokenized_datasets = dataset["train"].map(preprocess_function, batched=True)

Map:   0%|          | 0/1659083 [00:00<?, ? examples/s]

In [ ]:
# ---------------------------------------------------
# Step 6: Create Data Collator
# ---------------------------------------------------
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_checkpoint)

In [ ]:
# ---------------------------------------------------
# Step 7: Load Pretrained Model
# ---------------------------------------------------
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

pytorch_model.bin:   0%|          | 0.00/306M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [ ]:
# ---------------------------------------------------
# Step 8: Define Metrics (BLEU Score)
# ---------------------------------------------------
metric = evaluate.load("sacrebleu")

model.safetensors:   0%|          | 0.00/306M [00:00<?, ?B/s]

In [ ]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute BLEU
    result = metric.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    return {"bleu": result["score"]}

In [ ]:
# ---------------------------------------------------
# Step 9: Training Setup
# ---------------------------------------------------
# Note: For demonstration, we use a small subset and fewer epochs
train_dataset = tokenized_datasets.shuffle(seed=42).select(range(20000))  # subset for faster training
eval_dataset = tokenized_datasets.shuffle(seed=42).select(range(2000))

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch", # Changed from evaluation_strategy
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,    # increase to 3–5 for real training
    predict_with_generate=True,
    fp16=True,
    report_to="none", # Add this line to prevent logging to Weights & Biases
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# ---------------------------------------------------
# Step 10: Train Model (Fine-tuning)
# ---------------------------------------------------
trainer.train()

Epoch,Training Loss,Validation Loss,Bleu
1,2.579100,2.039850,39.103710
2,2.298400,1.884441,40.884627
3,2.147500,1.829279,41.284080


TrainOutput(global_step=7500, training_loss=2.286312548828125, metrics={'train_runtime': 1147.2053, 'train_samples_per_second': 52.301, 'train_steps_per_second': 6.538, 'total_flos': 782797992099840.0, 'train_loss': 2.286312548828125, 'epoch': 3.0})

In [ ]:
# ---------------------------------------------------
# Step 11: Test Translation
# ---------------------------------------------------
test_sentences = [
    "How are you?",
    "This is my first experiment with machine translation.",
    "I love learning artificial intelligence.",
]

inputs = tokenizer(test_sentences, return_tensors="pt", padding=True, truncation=True)

# Move the input tensors to the same device as the model
device = model.device
inputs = {name: tensor.to(device) for name, tensor in inputs.items()}

outputs = model.generate(**inputs)

print("Sample Translations (English → Hindi):\n")
for i, sentence in enumerate(test_sentences):
    print(f"EN: {sentence}")
    print(f"HI: {tokenizer.decode(outputs[i], skip_special_tokens=True)}\n")

Sample Translations (English → Hindi):

EN: How are you?
HI: तुम कैसे हो?

EN: This is my first experiment with machine translation.
HI: यह मशीन अनुवाद के साथ मेरा पहला प्रयोग है।

EN: I love learning artificial intelligence.
HI: मुझे कृत्रिम बुद्धि सीखना अच्छा लगता है।



In [ ]:
# ---------------------------------------------------
# Step 12: Interactive Translation (EN -> HI)
# ---------------------------------------------------
import torch

# Ensure model is on the correct device
model.to(device)
model.eval()

def translate_en2hi(text, max_new_tokens=64, num_beams=5):
    if not text.strip():
        return ""
    enc = tokenizer([text], return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        gen = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            early_stopping=True
        )
    return tokenizer.decode(gen[0], skip_special_tokens=True)

print("\nInteractive translator ready. Type /q to quit.\n")
while True:
    en = input("EN> ").strip()
    if en.lower() in {"/q", "quit", "exit"}:
        print("Bye!")
        break
    hi = translate_en2hi(en)
    print(f"HI> {hi}\n")


Interactive translator ready. Type /q to quit.

EN> I went to college yesterday.
HI> मैं कल कॉलेज गया।

EN> I love you
HI> मैं आपसे प्यार करता हूँ

EN> I love playing cricket
HI> मुझे क्रिकेट खेलना बहुत पसंद है।

EN> Who are you?
HI> तुम कौन हो?

EN> Wow what a beauty
HI> वाह क्या एक सौंदर्य

EN> Oh no! I missed the train
HI> ओह नहीं!

EN> I missed the train
HI> मैं ट्रेन छूटी

EN> He is better than me
HI> वह मुझसे बेहतर है

EN> She is cute and funny
HI> वह प्यारा और हास्यास्पद है।

EN> Where is Mumbai?
HI> मुंबई कहां है?

EN> India is the best country in the world
HI> भारत विश्व का सर्वोत्तम देश है।

EN> The exam was tough
HI> परीक्षा कठिन थी

EN> I will never hurt you again
HI> मैं फिर कभी आप को चोट नहीं पहुँचाएगा

EN> My mother cook food for us
HI> मेरी मां हमारे लिए खाना बनाती है

EN> Modi is the Prime Minister of India
HI> मोदी भारत का प्रधानमंत्री है।

EN> I am a certified Data Scientist
HI> मैं एक प्रमाणित आंकड़ा वैज्ञानिक हूँ

EN> I am a huge fan of Salmaan
HI> मैं सालान का एक

# Experiment 4

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# in a Colab cell
!pip install -U transformers datasets

In [ ]:
import argparse, os, random, math
from typing import List, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
# keep Trainer from prompting for wandb
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# seeds
torch.manual_seed(42); random.seed(42); np.random.seed(42)

In [ ]:
# ----------------------------
# Tiny corpus (for MLM demos)
# ----------------------------
def tiny_corpus() -> Tuple[List[List[str]], List[str]]:
    sents = [
        "i like machine learning",
        "i love deep learning",
        "this movie is great",
        "this film is terrible",
        "python is a good language",
        "programming in python is fun",
        "cats are wonderful pets",
        "dogs are friendly animals",
    ]
    tokenized = [s.split() for s in sents]
    tokens = [t for sent in tokenized for t in sent]
    return tokenized, tokens

In [ ]:
# ----------------------------
# Vocab utilities
# ----------------------------
class Vocab:
    def __init__(self, tokens: List[str], unk_token='<unk>', pad_token='<pad>', mask_token='[MASK]'):
        self.unk_token, self.pad_token, self.mask_token = unk_token, pad_token, mask_token
        uniq = [pad_token, unk_token, mask_token] + sorted(set(tokens))
        self.idx2tok = uniq
        self.tok2idx = {t:i for i,t in enumerate(self.idx2tok)}
        self.special_ids = {self.tok2idx[pad_token], self.tok2idx[unk_token], self.tok2idx[mask_token]}
    def __len__(self): return len(self.idx2tok)
    def encode(self, tokens: List[str]): return [self.tok2idx.get(t, self.tok2idx[self.unk_token]) for t in tokens]
    def decode(self, ids: List[int]): return [self.idx2tok[i] for i in ids]

In [ ]:
def collate_batch(batch: List[List[int]], pad_idx: int) -> torch.Tensor:
    maxlen = max(len(x) for x in batch)
    out = torch.full((len(batch), maxlen), pad_idx, dtype=torch.long)
    for i, seq in enumerate(batch):
        out[i, :len(seq)] = torch.tensor(seq, dtype=torch.long)
    return out

In [ ]:
# ---------------------------------------
# Masking (MLM) — no random token noise
# ---------------------------------------
def mask_batch(batch_ids: List[List[int]], vocab: Vocab, mask_prob=0.5):
    """
    For toy learning: 50% mask prob to expose more supervised tokens.
    90% -> [MASK], 10% keep original. No random-token replacement.
    """
    input_ids, labels = [], []
    for seq in batch_ids:
        seq_in = seq.copy()
        seq_labels = [-100] * len(seq)
        for i, tok in enumerate(seq):
            if random.random() < mask_prob:
                seq_labels[i] = tok
                if random.random() < 0.9:
                    seq_in[i] = vocab.tok2idx[vocab.mask_token]
        input_ids.append(seq_in)
        labels.append(seq_labels)
    return input_ids, labels

In [ ]:
# ============================================================
# 1) MLM FROM SCRATCH (window MLP) + test
# ============================================================
class WindowMLM(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, ctx=2):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.ctx = ctx
        self.proj = nn.Sequential(
            nn.Linear(emb_dim*(2*ctx), 128),
            nn.ReLU(),
            nn.Linear(128, vocab_size)
        )
    def forward(self, x):
        B,L = x.shape
        em = self.emb(x)  # (B,L,E)
        out = torch.zeros(B,L,self.proj[-1].out_features, device=x.device)
        pad = self.ctx
        em_pad = F.pad(em, (0,0,pad,pad))
        for i in range(L):
            left  = em_pad[:, i:i+pad, :].reshape(B, -1)
            right = em_pad[:, i+pad+1:i+2*pad+1, :].reshape(B, -1)
            ctx_vec = torch.cat([left, right], dim=1)
            out[:, i, :] = self.proj(ctx_vec)
        return out

In [ ]:
class RepeatedMLMDataset(Dataset):
    """Repeat the tiny corpus N times to create enough masked targets per epoch."""
    def __init__(self, tokenized_sents, vocab: Vocab, repeats: int = 200):
        base = [vocab.encode(s) for s in tokenized_sents]
        self.seqs = base * repeats
        self.vocab = vocab
    def __len__(self): return len(self.seqs)
    def __getitem__(self, idx): return self.seqs[idx]

In [ ]:
def train_mlm_from_scratch(epochs=8, batch_size=32, lr=1e-3, clip=1.0, repeats=200, mask_prob=0.5):
    print(f"=== Training MLM-from-scratch | epochs={epochs}, bs={batch_size}, lr={lr}, repeats={repeats}, mask_p={mask_prob}")
    tokenized, tokens = tiny_corpus()
    vocab = Vocab(tokens)
    ds = RepeatedMLMDataset(tokenized, vocab, repeats=repeats)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True,
                    collate_fn=lambda b: collate_batch(b, vocab.tok2idx[vocab.pad_token]))
    model = WindowMLM(len(vocab), emb_dim=64, ctx=2)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss()  # we will filter to masked positions ourselves

    model.train()
    for ep in range(epochs):
        total, steps = 0.0, 0
        for batch in dl:
            input_ids = batch.tolist()
            masked, labels = mask_batch(input_ids, vocab, mask_prob=mask_prob)
            inp = collate_batch(masked, vocab.tok2idx[vocab.pad_token])
            lab = collate_batch(labels, -100)  # will filter
            logits = model(inp)       # (B,L,V)
            B,L,V = logits.shape
            # --- compute loss ONLY on masked positions ---
            mask_sel = (lab.view(-1) != -100)
            if mask_sel.sum() == 0:
                continue
            logits_m = logits.view(-1, V)[mask_sel]
            labels_m = lab.view(-1)[mask_sel]
            loss = loss_fn(logits_m, labels_m)
            if not torch.isfinite(loss):
                print("WARNING: NaN/Inf loss — skip step"); continue
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
            total += loss.item(); steps += 1
        avg = total / max(1, steps)
        print(f"Epoch {ep+1}/{epochs} | masked-CE={avg:.4f}")
    return model, vocab

In [ ]:
def test_mlm_from_scratch(model: WindowMLM, vocab: Vocab, sentence="i love [MASK] learning"):
    print("\n[TEST] MLM-from-scratch")
    toks = sentence.split(); assert "[MASK]" in toks, "Include [MASK] in the sentence"
    ids = vocab.encode(toks)
    inp = collate_batch([ids], vocab.tok2idx[vocab.pad_token])
    with torch.no_grad():
        logits = model(inp).clone()
        # prevent specials
        for sid in vocab.special_ids: logits[0, :, sid] = -1e9
        mpos = toks.index("[MASK]")
        pred_id = int(logits[0, mpos].argmax())
        pred_tok = vocab.idx2tok[pred_id]
    print("Input :", sentence)
    print("Predicted token:", pred_tok)
    print("Filled sentence:", " ".join([t if t!="[MASK]" else pred_tok for t in toks]))

In [ ]:
# ============================================================
# 2) Transformer MLM + test
# ============================================================
class TransformerMLM(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, nhead=4, nlayers=2, dim_feedforward=256, max_len=128):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, emb_dim)
        self.pos_emb = nn.Embedding(max_len, emb_dim)
        try:
            enc_layer = nn.TransformerEncoderLayer(d_model=emb_dim, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   activation='gelu', batch_first=True)
            self.batch_first=True
        except TypeError:
            enc_layer = nn.TransformerEncoderLayer(d_model=emb_dim, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   activation='gelu')
            self.batch_first=False
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=nlayers)
        self.head = nn.Linear(emb_dim, vocab_size)
    def forward(self, input_ids, attention_mask=None):
        B,L = input_ids.shape
        pos = torch.arange(0, L, device=input_ids.device).unsqueeze(0).expand(B,L)
        x = self.tok_emb(input_ids) + self.pos_emb(pos)
        if self.batch_first:
            src = x
            src_key_padding_mask = (attention_mask==0) if attention_mask is not None else None
            out = self.encoder(src, src_key_padding_mask=src_key_padding_mask)
        else:
            src = x.permute(1,0,2)
            src_key_padding_mask = (attention_mask==0) if attention_mask is not None else None
            out = self.encoder(src, src_key_padding_mask=src_key_padding_mask).permute(1,0,2)
        return self.head(out)

In [ ]:
class RepeatedPadDataset(Dataset):
    def __init__(self, tokenized_sents, vocab: Vocab, repeats: int = 200):
        base = [vocab.encode(s) for s in tokenized_sents]
        self.seqs = base * repeats
        self.vocab = vocab
    def __len__(self): return len(self.seqs)
    def __getitem__(self, idx): return self.seqs[idx]

In [ ]:
def train_mlm_transformer(epochs=8, batch_size=32, lr=1e-4, clip=1.0, repeats=200, mask_prob=0.5):
    print(f"\n=== Training Transformer-MLM | epochs={epochs}, bs={batch_size}, lr={lr}, repeats={repeats}, mask_p={mask_prob}")
    tokenized, tokens = tiny_corpus()
    vocab = Vocab(tokens)
    ds = RepeatedPadDataset(tokenized, vocab, repeats=repeats)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True,
                    collate_fn=lambda b: collate_batch(b, vocab.tok2idx[vocab.pad_token]))
    model = TransformerMLM(len(vocab), emb_dim=128, nhead=4, nlayers=2, max_len=64)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss()  # we filter to masked positions

    model.train()
    for ep in range(epochs):
        total, steps = 0.0, 0
        for batch in dl:
            input_ids = batch.tolist()
            masked, labels = mask_batch(input_ids, vocab, mask_prob=mask_prob)
            inp = collate_batch(masked, vocab.tok2idx[vocab.pad_token])
            lab = collate_batch(labels, -100)
            attn = (inp != vocab.tok2idx[vocab.pad_token]).long()
            logits = model(inp, attention_mask=attn)
            B,L,V = logits.shape
            mask_sel = (lab.view(-1) != -100)
            if mask_sel.sum() == 0:
                continue
            loss = loss_fn(logits.view(-1, V)[mask_sel], lab.view(-1)[mask_sel])
            if not torch.isfinite(loss):
                print("WARNING: NaN/Inf loss — skip step"); continue
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
            total += loss.item(); steps += 1
        avg = total / max(1, steps)
        print(f"Epoch {ep+1}/{epochs} | masked-CE={avg:.4f}")
    return model, vocab

In [ ]:
def test_mlm_transformer(model: TransformerMLM, vocab: Vocab, sentence="this film is [MASK]"):
    print("\n[TEST] Transformer-MLM")
    toks = sentence.split(); assert "[MASK]" in toks, "Include [MASK] in the sentence"
    ids = vocab.encode(toks)
    inp = collate_batch([ids], vocab.tok2idx[vocab.pad_token])
    attn = (inp != vocab.tok2idx[vocab.pad_token]).long()
    with torch.no_grad():
        logits = model(inp, attention_mask=attn).clone()
        for sid in vocab.special_ids: logits[0, :, sid] = -1e9
        mpos = toks.index("[MASK]")
        pred_id = int(logits[0, mpos].argmax())
        pred_tok = vocab.idx2tok[pred_id]
    print("Input :", sentence)
    print("Predicted token:", pred_tok)
    print("Filled sentence:", " ".join([t if t!="[MASK]" else pred_tok for t in toks]))

In [ ]:
# ============================================================
# 3) BERT classification (balanced synthetic) + test
# ============================================================
def make_balanced_classification_set(n_per_class=150):
    pos_templates = [
        "this movie is great","i love deep learning","i like machine learning",
        "python is a good language","cats are wonderful pets","dogs are friendly animals",
        "the product works well","service was excellent","the tutorial is helpful"
    ]
    neg_templates = [
        "this film is terrible","the result is disappointing","the interface is confusing",
        "i hate this approach","the model fails often","support was awful",
        "performance is poor","the app keeps crashing","bad experience overall"
    ]
    neu_templates = [
        "the book is on the table","the system was updated yesterday","it is raining today",
        "this is a sample sentence","we scheduled a meeting","the code compiled successfully",
        "documentation is available","the data was recorded","the item shipped today"
    ]
    def expand(lst, k):
        out=[]
        while len(out)<k: out+=lst
        return out[:k]
    pos=expand(pos_templates,n_per_class); neg=expand(neg_templates,n_per_class); neu=expand(neu_templates,n_per_class)
    texts = pos+neg+neu
    labels = (["positive"]*len(pos))+ (["negative"]*len(neg))+ (["neutral"]*len(neu))
    return texts, labels

In [ ]:
def finetune_bert_and_test(epochs=4, batch_size=8, lr=2e-5, grad_accum=1, n_per_class=150, test_sentence="python is a good language"):
    try:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, set_seed
        from datasets import Dataset
        from sklearn.model_selection import train_test_split
        from sklearn.preprocessing import LabelEncoder
        import sklearn.metrics as skm
    except Exception:
        raise RuntimeError("Install: pip install transformers datasets scikit-learn")
    set_seed(42)
    texts, labels = make_balanced_classification_set(n_per_class=n_per_class)
    from sklearn.model_selection import train_test_split
    Xtr, Xte, ytr, yte = train_test_split(texts, labels, test_size=0.2, random_state=42, stratify=labels)
    le=LabelEncoder(); ytr_id=le.fit_transform(ytr); yte_id=le.transform(yte)
    train_ds=Dataset.from_dict({"text":Xtr,"labels":ytr_id})
    test_ds =Dataset.from_dict({"text":Xte,"labels":yte_id})
    tok = AutoTokenizer.from_pretrained("bert-base-uncased")
    def t(batch): return tok(batch["text"], truncation=True, padding="max_length", max_length=128)
    train_ds=train_ds.map(t, batched=True); test_ds=test_ds.map(t, batched=True)
    train_ds.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
    test_ds.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
    model=AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=len(le.classes_))
    args=TrainingArguments(
        output_dir="bert_out",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=max(1,batch_size),
        gradient_accumulation_steps=max(1,grad_accum),
        learning_rate=lr,
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=50,
        seed=42,
    )
    def metrics(ev):
        preds, labels_np = ev
        preds = np.argmax(preds, axis=-1)
        return {
            "accuracy": skm.accuracy_score(labels_np, preds),
            "f1_macro": skm.f1_score(labels_np, preds, average="macro"),
            "f1_weighted": skm.f1_score(labels_np, preds, average="weighted"),
        }
    trainer=Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds, tokenizer=tok, compute_metrics=metrics)
    print(f"\n=== Fine-tuning BERT | epochs={epochs}, bs={batch_size}, grad_accum={grad_accum}, lr={lr}, n_per_class={n_per_class}")
    trainer.train()
    print("Eval:", trainer.evaluate())
    # Test sentence
    enc = tok(test_sentence, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
    with torch.no_grad(): out=model(**enc); pred=int(out.logits.argmax(-1))
    inv={i:c for i,c in enumerate(le.classes_)}
    print("\n[TEST] BERT classification")
    print("Text:", test_sentence)
    print("Predicted label:", inv.get(pred, str(pred)))

In [ ]:
# ----------------------------
# CLI
# ----------------------------
if __name__ == "__main__":
    p=argparse.ArgumentParser()
    p.add_argument("--which", choices=["mlm_scratch","mlm_transformer","bert_finetune","all"], default="all")
    # scratch MLM
    p.add_argument("--mlm_epochs", type=int, default=8)
    p.add_argument("--mlm_bs", type=int, default=32)
    p.add_argument("--mlm_lr", type=float, default=1e-3)
    p.add_argument("--mlm_repeats", type=int, default=200)
    p.add_argument("--mlm_maskp", type=float, default=0.5)
    # transformer MLM
    p.add_argument("--trans_epochs", type=int, default=8)
    p.add_argument("--trans_bs", type=int, default=32)
    p.add_argument("--trans_lr", type=float, default=1e-4)
    p.add_argument("--trans_repeats", type=int, default=200)
    p.add_argument("--trans_maskp", type=float, default=0.5)
    # BERT
    p.add_argument("--bert_epochs", type=int, default=4)
    p.add_argument("--batch_size", type=int, default=8)
    p.add_argument("--grad_accum", type=int, default=2)
    p.add_argument("--lr_bert", type=float, default=2e-5)
    p.add_argument("--cls_n_per_class", type=int, default=150)
    # test sentences
    p.add_argument("--mask_sent_1", type=str, default="i love [MASK] learning")
    p.add_argument("--mask_sent_2", type=str, default="this film is [MASK]")
    p.add_argument("--cls_sent", type=str, default="python is a good language")
    args,_=p.parse_known_args()

    if args.which in ("mlm_scratch","all"):
        m1,v1 = train_mlm_from_scratch(epochs=args.mlm_epochs, batch_size=args.mlm_bs, lr=args.mlm_lr,
                                       repeats=args.mlm_repeats, mask_prob=args.mlm_maskp)
        test_mlm_from_scratch(m1, v1, args.mask_sent_1)

    if args.which in ("mlm_transformer","all"):
        m2,v2 = train_mlm_transformer(epochs=args.trans_epochs, batch_size=args.trans_bs, lr=args.trans_lr,
                                      repeats=args.trans_repeats, mask_prob=args.trans_maskp)
        test_mlm_transformer(m2, v2, args.mask_sent_2)

    if args.which in ("bert_finetune","all"):
        finetune_bert_and_test(epochs=args.bert_epochs, batch_size=args.batch_size, lr=args.lr_bert,
                               grad_accum=args.grad_accum, n_per_class=args.cls_n_per_class,
                               test_sentence=args.cls_sent)

=== Training MLM-from-scratch | epochs=8, bs=32, lr=0.001, repeats=200, mask_p=0.5
Epoch 1/8 | masked-CE=1.9536
Epoch 2/8 | masked-CE=0.5847
Epoch 3/8 | masked-CE=0.4424
Epoch 4/8 | masked-CE=0.3819
Epoch 5/8 | masked-CE=0.3478
Epoch 6/8 | masked-CE=0.3622
Epoch 7/8 | masked-CE=0.3416
Epoch 8/8 | masked-CE=0.3351

[TEST] MLM-from-scratch
Input : i love [MASK] learning
Predicted token: deep
Filled sentence: i love deep learning

=== Training Transformer-MLM | epochs=8, bs=32, lr=0.0001, repeats=200, mask_p=0.5
Epoch 1/8 | masked-CE=2.9401
Epoch 2/8 | masked-CE=1.9874
Epoch 3/8 | masked-CE=1.3566
Epoch 4/8 | masked-CE=0.9356
Epoch 5/8 | masked-CE=0.6873
Epoch 6/8 | masked-CE=0.5152
Epoch 7/8 | masked-CE=0.4135
Epoch 8/8 | masked-CE=0.3863

[TEST] Transformer-MLM
Input : this film is [MASK]
Predicted token: terrible
Filled sentence: this film is terrible


Map:   0%|          | 0/360 [00:00<?, ? examples/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).



=== Fine-tuning BERT | epochs=4, bs=8, grad_accum=2, lr=2e-05, n_per_class=150


Step,Training Loss
50,0.728600


Eval: {'eval_loss': 0.04512268304824829, 'eval_accuracy': 1.0, 'eval_f1_macro': 1.0, 'eval_f1_weighted': 1.0, 'eval_runtime': 40.3249, 'eval_samples_per_second': 2.232, 'eval_steps_per_second': 0.298, 'epoch': 4.0}

[TEST] BERT classification
Text: python is a good language
Predicted label: positive


# Experiment 5

**Part 1: Implementing a Vanilla Autoregressive Model from Scratch**

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [ ]:
# --- Step 1.1: Data Preparation ---
# Sample text data for training our model
text = "hello world. this is a simple autoregressive model. we will teach it to generate text character by character."

In [ ]:
# Create character vocabulary
chars = sorted(list(set(text)))
char_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_char = {i: ch for i, ch in enumerate(chars)}
vocab_size = len(chars)

In [ ]:
print("--- Data Preparation ---")
print(f"Original text has {len(text)} characters.")
print(f"Vocabulary size: {vocab_size}")
print(f"Vocabulary: {''.join(chars)}")
print("-" * 20)

--- Data Preparation ---
Original text has 109 characters.
Vocabulary size: 23
Vocabulary:  .abcdeghilmnoprstuvwxy
--------------------


In [ ]:
# Prepare input and target sequences
seq_length = 10
dataX = []
dataY = []
for i in range(0, len(text) - seq_length, 1):
    seq_in = text[i:i + seq_length]
    seq_out = text[i + 1:i + seq_length + 1]
    dataX.append([char_to_int[char] for char in seq_in])
    dataY.append([char_to_int[char] for char in seq_out])

In [ ]:
n_patterns = len(dataX)
print(f"Total Patterns (sequences): {n_patterns}")

Total Patterns (sequences): 99


In [ ]:
# Convert to PyTorch tensors
X = torch.tensor(dataX, dtype=torch.float32).reshape(n_patterns, seq_length, 1)
y = torch.tensor(dataY, dtype=torch.long)

In [ ]:
# Normalize inputs to be between 0-1
X = X / float(vocab_size)

In [ ]:
# --- Step 1.2: Model Definition (GRU) ---
class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        # Embedding layer is often used, but for this simple char model, we'll skip it.
        # The input is just the normalized character index.
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        output, hidden = self.gru(x, hidden)
        # We want to predict a character for each input character's timestep
        output = self.fc(output)
        return output, hidden

    def init_hidden(self, batch_size=1):
        # Initialize hidden state with zeros
        return torch.zeros(1, batch_size, self.hidden_size)

In [ ]:
# Hyperparameters
hidden_size = 64
learning_rate = 0.01
epochs = 10000

In [ ]:
model = CharRNN(input_size=1, hidden_size=hidden_size, output_size=vocab_size)
print("--- Model Architecture ---")
print(model)
print("-" * 20)

--- Model Architecture ---
CharRNN(
  (gru): GRU(1, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=23, bias=True)
)
--------------------


In [ ]:
# --- Step 1.3: Training the Model ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
print("--- Starting Training ---")
for epoch in range(epochs):
    # For simplicity, we process the whole dataset as one batch
    hidden = model.init_hidden(batch_size=X.shape[0])

    # Forward pass
    outputs, hidden = model(X, hidden)

    # Loss calculation
    # Reshape outputs and y to match CrossEntropyLoss expectations
    # [batch_size, seq_length, vocab_size] -> [batch_size * seq_length, vocab_size]
    loss = criterion(outputs.view(-1, vocab_size), y.view(-1))

    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 1000 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

--- Starting Training ---
Epoch [1000/10000], Loss: 0.3067
Epoch [2000/10000], Loss: 0.2635
Epoch [3000/10000], Loss: 0.2511
Epoch [4000/10000], Loss: 0.2360
Epoch [5000/10000], Loss: 0.2217
Epoch [6000/10000], Loss: 0.2283
Epoch [7000/10000], Loss: 0.2184
Epoch [8000/10000], Loss: 0.2121
Epoch [9000/10000], Loss: 0.2230
Epoch [10000/10000], Loss: 0.2126


In [ ]:
# --- Step 1.4: Generating Text (Autoregressive Inference) ---
def generate_text(model, start_string, length=50):
    print("--- Generating Text ---")
    print(f'Seed: "{start_string}"')

    model.eval() # Set model to evaluation mode
    with torch.no_grad():
        # Prepare the initial input sequence
        chars_input = [char_to_int[c] for c in start_string]
        input_seq = torch.tensor(chars_input, dtype=torch.float32).reshape(1, len(start_string), 1)
        input_seq = input_seq / float(vocab_size)

        generated_text = start_string
        hidden = model.init_hidden(batch_size=1)

        # "Warm up" the hidden state with the seed string
        _, hidden = model(input_seq, hidden)

        # The last character of the seed is the first input for generation
        last_char_input = input_seq[:, -1, :].unsqueeze(1)

        # Autoregressively generate text
        for _ in range(length):
            output, hidden = model(last_char_input, hidden)
            # Get probabilities and find the most likely character
            _, top_i = output.topk(1)
            char_index = top_i[0].item()

            # Append predicted character to the text
            char = int_to_char[char_index]
            generated_text += char

            # Use the predicted character as the next input
            last_char_input = torch.tensor([[char_index]], dtype=torch.float32).reshape(1, 1, 1)
            last_char_input = last_char_input / float(vocab_size)

    return generated_text

In [ ]:
# --- Step 1.5: Run the Generation ---
final_text = generate_text(model, start_string="hello", length=100)
print("Generated Text:")
print(final_text)

--- Generating Text ---
Seed: "hello"
Generated Text:
hello vo geaeaawelt is a simple autoregressive model. we will teach it to generate text ch racter by char


**Part 2 — Compare AR with GPT on a summarization benchmark**

In [ ]:
# --- Step 2.1: Install and Import Libraries ---
# Run these commands in your Colab notebook cell
!pip install transformers datasets evaluate rouge_score -q

In [ ]:
import torch
from datasets import load_dataset
from transformers import T5ForConditionalGeneration, T5Tokenizer
import evaluate

In [ ]:
# --- Step 2.2: Load Dataset and Model ---
# Use a smaller, well-known dataset for summarization to run quickly in Colab
dataset_name = "cnn_dailymail" # Changed from "samsum"
dataset_config = "3.0.0" # Added config for cnn_dailymail
# Use t5-small, a powerful yet manageable transformer for summarization
model_name = "t5-small"

In [ ]:
# Load the dataset (we'll just use a few examples from the test set)
dataset = load_dataset(dataset_name, dataset_config, split="test") # Added dataset_config
# Load the ROUGE metric for evaluation
rouge_metric = evaluate.load("rouge")

In [ ]:
# Load pre-trained tokenizer and model
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

print("\n--- Loaded Dataset and Pre-trained T5-small Model ---")


# --- Step 2.3: Generate Summaries for a Few Examples ---
# Let's take 3 examples from the test set
# The key for the text is 'article' and for the summary is 'highlights' for cnn_dailymail
sample_articles = [dataset[i]['article'] for i in range(3)] # Changed key from 'dialogue' to 'article'
reference_summaries = [dataset[i]['highlights'] for i in range(3)] # Changed key from 'summary' to 'highlights'
generated_summaries = []

# The prompt for T5 needs to be "summarize: <article>"
for article in sample_articles:
    prompt = f"summarize: {article}"

    # Tokenize the input
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids

    # Generate the summary autoregressively using the model's .generate() method
    output_ids = model.generate(
        input_ids,
        max_length=80,      # Set a max length for the summary
        num_beams=4,        # Use beam search for better results
        early_stopping=True # Stop when the model is done
    )

    # Decode the generated tokens back to a string
    summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    generated_summaries.append(summary)

# --- Step 2.4: Display and Evaluate Performance ---
print("\n--- Summarization Results ---")
for i in range(len(sample_articles)):
    print(f"--- Example {i+1} ---")
    print(f"\nOriginal Dialogue:\n{sample_articles[i]}") # Still printing as "Original Dialogue" for consistency with original code output format
    print(f"\nReference Summary:\n{reference_summaries[i]}")
    print(f"\nModel Generated Summary:\n{generated_summaries[i]}")
    print("-" * 30)

# Calculate ROUGE scores
results = rouge_metric.compute(predictions=generated_summaries, references=reference_summaries)

print("\n--- ROUGE Score Evaluation ---")
print("This metric compares the generated summaries to the reference summaries.")
for key, value in results.items():
    print(f"{key.upper()}: {value*100:.2f}%")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (789 > 512). Running this sequence through the model will result in indexing errors



--- Loaded Dataset and Pre-trained T5-small Model ---

--- Summarization Results ---
--- Example 1 ---

Original Dialogue:
(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the 

# Experiment 6

**Cell 1: Install Dependencies**

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers pandas gradio pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


**Cell 2: Import Libraries and Load Data**

In [ ]:
import pandas as pd
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
import warnings

In [ ]:
# Suppress warnings
warnings.filterwarnings('ignore')

In [ ]:
# Define file paths
faculty_file = '/content/faculty.csv'
bulletin_file = '/content/academic_bulletin.csv'

In [ ]:
# Load the CSV files with 'latin1' encoding
try:
    df_faculty = pd.read_csv(faculty_file, encoding='latin1')
    df_bulletin = pd.read_csv(bulletin_file, encoding='latin1')

    print("--- Faculty Data (Loaded) ---")
    print(df_faculty.head())
    print("\n--- Academic Bulletin Data (Loaded) ---")
    print(df_bulletin.head())

except Exception as e:
    print(f"Error loading files: {e}")
    print("Please make sure 'faculty.csv' and 'academic_bulletin.csv' are uploaded to the '/content/' directory.")

--- Faculty Data (Loaded) ---
                                                 url                  name  \
0  https://www.djsce.ac.in/faculty-docs/Kriti_Sri...      Kriti Srivastava   
1  https://www.djsce.ac.in/faculty-docs/325_VTBio...        Nilesh Marathe   
2  https://www.djsce.ac.in/faculty-docs/284_VTBio...         Kanchan Dabre   
3  https://www.djsce.ac.in/faculty-docs/285_VTBio...          Pooja Vartak   
4  https://www.djsce.ac.in/faculty-docs/Biodata_A...  Mohammed Adil Shaikh   

    joined_on             designation  yoe  \
0  02.07.2007  Head of the Department   18   
1  16.12.2022     Associate Professor   22   
2  20.12.2021     Assistant Professor   12   
3  20.12.2021     Assistant Professor   11   
4  01.09.2023     Assistant Professor    4   

                                    area_of_interest  \
0  Computer Vision, Responsible AI, Computational...   
1  Data Science for Network Security, Computer Ne...   
2         Computer Vision, NLP, AI, Machine Learning   


**Cell 3: Prepare and Chunk Documents**

In [ ]:
# Create a list to hold all documents
all_docs = []

In [ ]:
# Process Faculty data
print("Processing faculty data...")
for index, row in df_faculty.iterrows():
    # Combine relevant columns into a single text string
    content = f"Faculty Name: {row['name']}\n" \
              f"Designation: {row['designation']}\n" \
              f"Joined On: {row['joined_on']}\n" \
              f"Years of Experience: {row['yoe']}\n" \
              f"Area of Interest: {row['area_of_interest']}\n" \
              f"Email: {row['email']}"

    # Create a Document object
    doc = Document(
        page_content=content,
        metadata={"source": "faculty.csv", "faculty_name": row['name']}
    )
    all_docs.append(doc)

Processing faculty data...


In [ ]:
# Process Academic Bulletin data
print("Processing academic bulletin data...")
for index, row in df_bulletin.iterrows():
    # Use the 'text' column as the main content
    content = row['text']

    # Create a Document object
    doc = Document(
        page_content=content,
        metadata={"source": "academic_bulletin.csv", "section": row['section_title']}
    )
    all_docs.append(doc)

Processing academic bulletin data...


In [ ]:
print(f"Total number of documents created: {len(all_docs)}")

Total number of documents created: 34


In [ ]:
# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Max characters per chunk
    chunk_overlap=50   # Characters to overlap between chunks
)

In [ ]:
# Split the documents into chunks
all_chunks = text_splitter.split_documents(all_docs)

In [ ]:
print(f"Total number of chunks created: {len(all_chunks)}")
print("\n--- Example Chunk ---")
print(all_chunks[0].page_content)
print(f"Metadata: {all_chunks[0].metadata}")

Total number of chunks created: 34

--- Example Chunk ---
Faculty Name: Kriti Srivastava
Designation: Head of the Department
Joined On: 02.07.2007
Years of Experience: 18
Area of Interest: Computer Vision, Responsible AI, Computational Neuroscience
Email: kriti.srivastava@djsce.ac.in
Metadata: {'source': 'faculty.csv', 'faculty_name': 'Kriti Srivastava'}


**Cell 4: Create Embeddings and Vector Store**

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
print("Loading embedding model...")
# Initialize the embedding model
# We use a popular, efficient model from Hugging Face
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model_kwargs = {'device': 'cpu'} # Use CPU, or 'cuda' if you have a GPU
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs
)

print(f"Embedding model '{model_name}' loaded.")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model 'sentence-transformers/all-MiniLM-L6-v2' loaded.


In [ ]:
# Create the FAISS vector store from our chunks
print("Creating FAISS vector store...")
try:
    vector_store = FAISS.from_documents(all_chunks, embeddings)
    print("FAISS vector store created successfully.")

    # Test the retriever
    print("\n--- Testing retriever ---")
    test_query = "Who is the Head of the Department?"
    retrieved_docs = vector_store.similarity_search(test_query, k=1) # Get top 1 relevant doc
    print(f"Query: '{test_query}'")
    print("Retrieved document content:")
    print(retrieved_docs[0].page_content)

except Exception as e:
    print(f"Error creating vector store: {e}")

Creating FAISS vector store...
FAISS vector store created successfully.

--- Testing retriever ---
Query: 'Who is the Head of the Department?'
Retrieved document content:
This section lists aggregated counts of the department's activities for June 2024 - May 2025.


**Cell 5: Load the LLM and Test (Non-RAG)**

In [ ]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import torch

In [ ]:
print("Loading local LLM (google/flan-t5-base)...")
print("This may take a few minutes to download the model.")

Loading local LLM (google/flan-t5-base)...
This may take a few minutes to download the model.


In [ ]:
# Define model name and tokenizer
model_id = "google/flan-t5-base" # A reasonably sized model for Colab
tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [ ]:
# Load the model
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
# Create a text-generation pipeline
pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=512,  # Max length of the generated answer
    dtype=torch.bfloat16, # Use bfloat16 for efficiency
    device_map="auto" # Automatically use GPU if available
)

Device set to use cpu


In [ ]:
# Create the LangChain LLM wrapper
llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
print("--- Testing the LLM (non-RAG) ---")
# This is the "non-RAG based model" comparison
non_rag_query = "Who is the Head of the Department at D.J. Sanghvi?"

--- Testing the LLM (non-RAG) ---


In [ ]:
# We ask the question *without* any retrieved context
response = llm.invoke(non_rag_query)

In [ ]:
print(f"Query: {non_rag_query}")
print(f"Non-RAG Response: {response}")

Query: Who is the Head of the Department at D.J. Sanghvi?
Non-RAG Response: arun naidu


In [ ]:
print("LLM loaded successfully.")

LLM loaded successfully.


**Cell 6: Create and Test RAG Pipeline**

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

In [ ]:
print("--- Building the RAG Pipeline ---")

--- Building the RAG Pipeline ---


In [ ]:
# First, we make our vector_store usable as a "retriever"
# This tells the chain how to fetch relevant documents
retriever = vector_store.as_retriever(
    search_type="similarity", # Use similarity search
    search_kwargs={'k': 2}     # Get the top 2 most relevant chunks
)

In [ ]:
# Define the prompt template
# This is the most important part!
# We instruct the LLM to use the provided {context} to answer the {question}.
prompt_template = """
Use the following pieces of context to answer the question at the end.
If you don't know the answer from the context, just say that you don't know, don't try to make up an answer.

{context}

Question: {question}
Helpful Answer:
"""

In [ ]:
# Create the prompt from the template
RAG_PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": RAG_PROMPT},
    return_source_documents=True # We also want to see which chunks were used
)

print("RAG chain created.")

RAG chain created.


In [ ]:
print("--- Testing the RAG Pipeline ---")

# Let's use a query that we know is in the data
rag_query = "Who is the Head of the Department?"

# Run the RAG chain
result = rag_chain.invoke(rag_query)

print(f"Query: {rag_query}")
print(f"RAG Response: {result['result']}")

print("\n--- Source Documents Used ---")
for doc in result['source_documents']:
    print("---")
    print(doc.page_content)
    print(f"Source: {doc.metadata['source']}")

--- Testing the RAG Pipeline ---
Query: Who is the Head of the Department?
RAG Response: Govind Pandurang Wakure

--- Source Documents Used ---
---
This section lists aggregated counts of the department's activities for June 2024 - May 2025.
Source: academic_bulletin.csv
---
Faculty Name: Govind Pandurang Wakure
Designation: Assistant Professor
Joined On: 01.01.2025
Years of Experience: 16
Area of Interest: Database, Web Development, Machine Learning, Deep Learning, Data Science
Email: govind.wakure@djsce.ac.in
Source: faculty.csv


In [ ]:
print("--- Debugging the Retriever ---")
print("Let's find the top 5 most similar documents to the query.")

debug_query = "Who is the Head of the Department?"
debug_docs = vector_store.similarity_search(debug_query, k=5) # Get top 5 relevant docs

for i, doc in enumerate(debug_docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Similarity Score (if available): {doc.metadata.get('score')}") # FAISS might not show score here
    print(doc.page_content)
    print(f"Source: {doc.metadata['source']}")

print("\n--- Analysis ---")
print("Check the 5 results above. We are looking for the document containing 'Kriti Srivastava' and 'Head of the Department'.")
print("If it's not in this list, or is very low, our retriever is not effective for this query.")

--- Debugging the Retriever ---
Let's find the top 5 most similar documents to the query.

--- Result 1 ---
Similarity Score (if available): None
This section lists aggregated counts of the department's activities for June 2024 - May 2025.
Source: academic_bulletin.csv

--- Result 2 ---
Similarity Score (if available): None
Faculty Name: Govind Pandurang Wakure
Designation: Assistant Professor
Joined On: 01.01.2025
Years of Experience: 16
Area of Interest: Database, Web Development, Machine Learning, Deep Learning, Data Science
Email: govind.wakure@djsce.ac.in
Source: faculty.csv

--- Result 3 ---
Similarity Score (if available): None
Dr. Kriti Srivastava â Head of Department; Prof. Pradnya Saval â Treasurer; Prof Pooja Vartak â Internship and Placement Coordinator; Lab in-charges: Computer Vision Lab: Prof. Adil Shaikh; Machine Learning Lab: Prof. Shruti Mathur; ...
Source: academic_bulletin.csv

--- Result 4 ---
Similarity Score (if available): None
Faculty Name: Kriti Srivasta

**Cell 7: Re-create RAG Pipeline with k=4**

In [ ]:
print("--- Re-building the RAG Pipeline with k=4 ---")

# We re-define the retriever to fetch the top 4 documents
retriever_k4 = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={'k': 4}  # <--- THIS IS THE FIX
)

# We use the same prompt template as before
prompt_template = """
Use the following pieces of context to answer the question at the end.
If you don't know the answer from the context, just say that you don't know, don't try to make up an answer.

{context}

Question: {question}
Helpful Answer:
"""

RAG_PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

# Create the new RAG chain using the new retriever
rag_chain_k4 = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever_k4, # Use the new retriever
    chain_type_kwargs={"prompt": RAG_PROMPT},
    return_source_documents=True
)

print("New RAG chain (k=4) created.")

print("\n--- Testing the (k=4) RAG Pipeline ---")

rag_query = "Who is the Head of the Department?"

# Run the new RAG chain
result = rag_chain_k4.invoke(rag_query)

print(f"Query: {rag_query}")
print(f"RAG Response: {result['result']}")

print("\n--- Source Documents Used (k=4) ---")
for doc in result['source_documents']:
    print("---")
    print(doc.page_content)
    print(f"Source: {doc.metadata['source']}")

--- Re-building the RAG Pipeline with k=4 ---
New RAG chain (k=4) created.

--- Testing the (k=4) RAG Pipeline ---
Query: Who is the Head of the Department?
RAG Response: Dr. Kriti Srivastava

--- Source Documents Used (k=4) ---
---
This section lists aggregated counts of the department's activities for June 2024 - May 2025.
Source: academic_bulletin.csv
---
Faculty Name: Govind Pandurang Wakure
Designation: Assistant Professor
Joined On: 01.01.2025
Years of Experience: 16
Area of Interest: Database, Web Development, Machine Learning, Deep Learning, Data Science
Email: govind.wakure@djsce.ac.in
Source: faculty.csv
---
Dr. Kriti Srivastava â Head of Department; Prof. Pradnya Saval â Treasurer; Prof Pooja Vartak â Internship and Placement Coordinator; Lab in-charges: Computer Vision Lab: Prof. Adil Shaikh; Machine Learning Lab: Prof. Shruti Mathur; ...
Source: academic_bulletin.csv
---
Faculty Name: Kriti Srivastava
Designation: Head of the Department
Joined On: 02.07.2007
Years of

**Cell 8: Build the Chatbot Interface (Gradio)**

In [ ]:
import gradio as gr

print("Building Gradio interface...")

# This function will be called by the Gradio interface
def answer_question(query):
    """
    Takes a user query, invokes the RAG chain, and formats the output.
    """
    try:
        # Run the query through our RAG chain
        result = rag_chain_k4.invoke(query)

        answer = result['result']

        # Format the source documents for display
        sources_text = ""
        for doc in result['source_documents']:
            sources_text += "--- SOURCE: " + doc.metadata.get('source', 'Unknown') + " ---\n"
            sources_text += doc.page_content + "\n\n"

        return answer, sources_text

    except Exception as e:
        print(f"Error in Gradio app: {e}")
        return f"An error occurred: {e}", ""

# Create the Gradio interface
# This defines the inputs and outputs
iface = gr.Interface(
    fn=answer_question,  # The function to call
    inputs=gr.Textbox(lines=2, placeholder="Ask a question about D.J. Sanghvi..."),
    outputs=[
        gr.Textbox(label="Answer"),
        gr.Textbox(label="Source Documents Used")
    ],
    title="D.J. Sanghvi (Data Science) RAG Chatbot",
    description="This bot uses a RAG system to answer questions based on the faculty and academic bulletin CSV files. "
                "Try asking: 'Who is the Head of the Department?' or 'Who is the coordinator for the internship?'",
    examples=[
        ["Who is the Head of the Department?"],
        ["Who is the internship and placement coordinator?"],
        ["What are the department's mission points?"],
        ["What are the areas of interest for Kanchan Dabre?"]
    ]
)

# Launch the app!
# share=True will create a public link (lasts 72 hours)
print("Launching Gradio app... (This will output a public link)")
iface.launch(share=True)

Building Gradio interface...
Launching Gradio app... (This will output a public link)
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fcb4a01c8b23b731ef.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Experiment 8

**Exercise 1**

In [ ]:
# ================================
# STEP 1 — Setup & Load Sample Data
# ================================
# Run this cell first in Google Colab.
# --- Installs (quiet) ---
!pip uninstall -y numpy
!pip -q install "numpy<2" transformers==4.45.0 datasets==2.20.0 evaluate==0.4.2 bert-score==0.3.13 accelerate==0.34.2 sentencepiece==0.2.0
# --- Imports & Config ---
import random
import torch
from datasets import load_dataset

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.35.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is i

In [ ]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cpu


In [ ]:
# --- Experiment Constants (edit if needed) ---
N_SAMPLES = 3                 # small, fast demo as per lab expectations
MAX_INPUT_TOKENS = 1024       # BART can handle up to 1024 tokens
MAX_SUMMARY_TOKENS = 200      # cap summary length

# --- Load CNN/DailyMail (test split), select a small random sample ---
ds = load_dataset("cnn_dailymail", "3.0.0", split="test")
ds = ds.shuffle(seed=SEED).select(range(N_SAMPLES))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [ ]:
# --- Quick preview of the chosen articles & reference summaries ---
for i, ex in enumerate(ds):
    print(f"\n=== SAMPLE {i+1} ===")
    print("[ARTICLE]:")
    print(ex["article"][:800].replace("\n", " ") + ("..." if len(ex["article"]) > 800 else ""))
    print("\n[REFERENCE SUMMARY]:")
    print(ex["highlights"].replace("\n", " "))

# --- Keep handy variables for next steps ---
ARTICLES = [ex["article"] for ex in ds]
REFERENCES = [ex["highlights"] for ex in ds]

print(f"\nLoaded {len(ARTICLES)} samples for summarization.")


=== SAMPLE 1 ===
[ARTICLE]:
(CNN) I see signs of a revolution everywhere. I see it in the op-ed pages of the newspapers, and on the state ballots in nearly half the country. I see it in politicians who once preferred to play it safe with this explosive issue but are now willing to stake their political futures on it. I see the revolution in the eyes of sterling scientists, previously reluctant to dip a toe into this heavily stigmatized world, who are diving in head first. I see it in the new surgeon general who cites data showing just how helpful it can be. I see a revolution in the attitudes of everyday Americans. For the first time a majority, 53%, favor its legalization, with 77% supporting it for medical purposes. Support for legalization has risen 11 points in the past few years alone. In 1969, the first time Pe...

[REFERENCE SUMMARY]:
CNN's Dr. Sanjay Gupta says we should legalize medical marijuana now . He says he knows how easy it is do nothing "because I did nothing for too 

In [ ]:
# ================================
# STEP 2 — Load Model & Generate Summaries
# ================================
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/bart-large-cnn"

In [ ]:
# --- Load Model + Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

print(f"Model {MODEL_NAME} loaded successfully.")

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Model facebook/bart-large-cnn loaded successfully.


In [ ]:
# --- Generate summaries ---
GENERATED_SUMMARIES = []
for i, article in enumerate(ARTICLES):
    inputs = tokenizer(
        article,
        max_length=MAX_INPUT_TOKENS,
        truncation=True,
        return_tensors="pt"
    ).to(DEVICE)

    summary_ids = model.generate(
        **inputs,
        num_beams=4,
        max_length=MAX_SUMMARY_TOKENS,
        early_stopping=True
    )

    summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    GENERATED_SUMMARIES.append(summary_text)

    print(f"\n=== GENERATED SUMMARY {i+1} ===")
    print(summary_text)
    print("\n[REFERENCE SUMMARY]:")
    print(REFERENCES[i])

print("\nAll summaries generated successfully.")


=== GENERATED SUMMARY 1 ===
CNN's John Sutter says he sees signs of a medical marijuana revolution everywhere. For the first time a majority, 53%, favor its legalization, he says. Support for legalization has risen 11 points in the past few years alone. Sutter: "Weed 3" will be the first federally approved clinical study on the use of marijuana for PTSD.

[REFERENCE SUMMARY]:
CNN's Dr. Sanjay Gupta says we should legalize medical marijuana now .
He says he knows how easy it is do nothing "because I did nothing for too long"

=== GENERATED SUMMARY 2 ===
Baby-faced boy from Memphis, Tennessee, has amassed more than 3,000 followers on Twitter. In many pictures he is smoking suspicious substances, with captions such as 'High Life' Tweets include the phrases, 'I need a bad b****', 'f*** da police', and 'gang sh** n****' He has prompted a wave of critics calling his stunts'sad'

[REFERENCE SUMMARY]:
Child has amassed thousands of Twitter followers with 'gang life' photos .
In one video he p

In [ ]:
# ================================
# STEP 3 — Evaluate Summaries (Final Fixed Version)
# ================================
!pip -q install sacrebleu rouge-score nltk bert-score

import sacrebleu
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from bert_score import score as bertscore_fn
import nltk

# --- Download required NLTK data ---
nltk.download("wordnet", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)   # 🔧 fix for latest NLTK

# (rest of your code stays unchanged below this)

# --- BLEU (SacreBLEU) ---
bleu = sacrebleu.corpus_bleu(GENERATED_SUMMARIES, [REFERENCES])
print("=== BLEU SCORE ===")
print(f"BLEU: {bleu.score:.4f}")

# --- ROUGE (rouge_score library) ---
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
rouge_scores = {"rouge1": 0, "rouge2": 0, "rougeL": 0}
for pred, ref in zip(GENERATED_SUMMARIES, REFERENCES):
    scores = scorer.score(ref, pred)
    for k in rouge_scores:
        rouge_scores[k] += scores[k].fmeasure
for k in rouge_scores:
    rouge_scores[k] /= len(GENERATED_SUMMARIES)
print("\n=== ROUGE SCORES ===")
for k, v in rouge_scores.items():
    print(f"{k}: {v:.4f}")

# --- METEOR (nltk, with tokenization fix) ---
meteor_scores = [
    meteor_score([word_tokenize(ref)], word_tokenize(pred))
    for pred, ref in zip(GENERATED_SUMMARIES, REFERENCES)
]
meteor_avg = sum(meteor_scores) / len(meteor_scores)
print("\n=== METEOR SCORE ===")
print(f"METEOR: {meteor_avg:.4f}")

# --- BERTSCORE ---
P, R, F1 = bertscore_fn(GENERATED_SUMMARIES, REFERENCES, lang="en", verbose=False)
print("\n=== BERTSCORE ===")
print(f"Precision: {P.mean().item():.4f}")
print(f"Recall:    {R.mean().item():.4f}")
print(f"F1 Score:  {F1.mean().item():.4f}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.4 MB/s eta 0:00:00
=== BLEU SCORE ===
BLEU: 2.9332

=== ROUGE SCORES ===
rouge1: 0.3218
rouge2: 0.0754
rougeL: 0.2050

=== METEOR SCORE ===
METEOR: 0.2582


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== BERTSCORE ===
Precision: 0.8539
Recall:    0.8638
F1 Score:  0.8588


**Exercise 2**

In [ ]:
# ================================
# STEP 1 — GLUE SST-2 Sentiment Classification
# ================================
from transformers import pipeline

# Load pre-trained sentiment model (fine-tuned on GLUE SST-2)
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [ ]:
# Few sample sentences for demonstration
samples_sst2 = [
    "The movie was absolutely fantastic, I loved every moment!",
    "It was a complete waste of time and money.",
    "The storyline was decent, but the acting felt flat."
]

# Run predictions
print("=== GLUE SST-2 Sentiment Classification ===")
for text in samples_sst2:
    result = sentiment_analyzer(text)[0]
    print(f"\nText: {text}")
    print(f"Predicted Label: {result['label']} | Confidence: {result['score']:.4f}")

=== GLUE SST-2 Sentiment Classification ===

Text: The movie was absolutely fantastic, I loved every moment!
Predicted Label: POSITIVE | Confidence: 0.9999

Text: It was a complete waste of time and money.
Predicted Label: NEGATIVE | Confidence: 0.9998

Text: The storyline was decent, but the acting felt flat.
Predicted Label: NEGATIVE | Confidence: 0.9995


In [ ]:
# ================================
# STEP 2 — SQuAD Question Answering
# ================================
from transformers import pipeline

# Load pre-trained QA model
qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad"
)

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
# Context passage (sample from SQuAD-like text)
context = """
The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.
It was named after the engineer Gustave Eiffel, whose company designed and built the tower.
Constructed from 1887 to 1889 as the entrance to the 1889 World's Fair,
it was initially criticized by some of France's leading artists and intellectuals for its design,
but it has become a global cultural icon of France and one of the most recognizable structures in the world.
"""

# Ask a few sample questions
questions = [
    "Who designed the Eiffel Tower?",
    "When was the Eiffel Tower constructed?",
    "Where is the Eiffel Tower located?"
]

print("=== SQuAD Question Answering ===")
for q in questions:
    result = qa_pipeline(question=q, context=context)
    print(f"\nQuestion: {q}")
    print(f"Answer: {result['answer']} (Score: {result['score']:.4f})")

=== SQuAD Question Answering ===

Question: Who designed the Eiffel Tower?
Answer: Gustave Eiffel (Score: 0.9916)

Question: When was the Eiffel Tower constructed?
Answer: 1887 to 1889 (Score: 0.2902)

Question: Where is the Eiffel Tower located?
Answer: Champ de Mars in Paris, France (Score: 0.4060)


In [ ]:
# ================================
# STEP 4 — TruthfulQA: Setup & Data
# ================================
!pip -q install datasets==2.20.0 transformers==4.45.0 bert-score==0.3.13 accelerate==0.34.2

import random, math, torch
from datasets import load_dataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# Load the TruthfulQA 'generation' task (free-form answers)
tqa = load_dataset("truthful_qa", "generation")

# Use a manageable sample size for a quick run; increase if you want
TQA_N = 50
tqa_eval = tqa["validation"].shuffle(seed=SEED).select(range(TQA_N))

# Convenience accessors
TQA_QUESTIONS = tqa_eval["question"]
TQA_CORRECT = tqa_eval["correct_answers"]      # list[str] per item
TQA_INCORRECT = tqa_eval["incorrect_answers"]  # list[str] per item

print(f"TruthfulQA loaded: {len(tqa_eval)} items for evaluation.")

Generating validation split:   0%|          | 0/817 [00:00<?, ? examples/s]

TruthfulQA loaded: 50 items for evaluation.


In [ ]:
# ================================
# STEP 5 — Choose Models to Compare
# ================================
# We'll compare two FLAN-T5 variants (instruction-following, light enough for Colab).
# You can swap with other text2text models if you want.

MODEL_IDS = [
    "google/flan-t5-base",
    "google/flan-t5-large",
]

DEVICE = 0 if torch.cuda.is_available() else -1  # pipeline device selector
print("Using device:", "CUDA" if DEVICE == 0 else "CPU")

Using device: CPU


In [ ]:
# ================================
# STEP 6 — Generate Answers on TruthfulQA
# ================================
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

def format_prompt(q: str) -> str:
    # Simple, explicit instruction helps FLAN-T5 be concise.
    return f"Answer factually and concisely:\nQ: {q}\nA:"

def generate_answers(model_id, questions, max_new_tokens=64, batch_size=8):
    tok = AutoTokenizer.from_pretrained(model_id)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(model_id).to("cuda" if DEVICE == 0 else "cpu")
    mdl.eval()

    answers = []
    for i in range(0, len(questions), batch_size):
        batch_q = questions[i:i+batch_size]
        prompts = [format_prompt(q) for q in batch_q]
        enc = tok(prompts, return_tensors="pt", padding=True, truncation=True).to(mdl.device)
        with torch.no_grad():
            out_ids = mdl.generate(
                **enc,
                do_sample=False,
                num_beams=4,
                max_new_tokens=max_new_tokens,
                early_stopping=True
            )
        batch_ans = tok.batch_decode(out_ids, skip_special_tokens=True)
        # Strip the leading "A:" if model parrots the format
        answers.extend([a.replace("A:", "", 1).strip() for a in batch_ans])

    return answers

MODEL_ANSWERS = {}
for mid in MODEL_IDS:
    print(f"Generating for {mid} ...")
    MODEL_ANSWERS[mid] = generate_answers(mid, TQA_QUESTIONS)
print({k: len(v) for k, v in MODEL_ANSWERS.items()})

Generating for google/flan-t5-base ...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

{'google/flan-t5-base': 50, 'google/flan-t5-large': 50}


In [ ]:
# ================================
# STEP 7 — Truthfulness Scoring (BERTScore)
# ================================
# Heuristic:
# For each question:
#   - Compute max BERTScore F1 similarity of the model answer vs any correct reference.
#   - Compute max BERTScore F1 similarity vs any incorrect reference.
#   - Truthful = (max_correct > max_incorrect); else Untruthful.
#
# This is a common, lightweight proxy. For a stricter evaluation, do manual labeling on a subset.

from bert_score import score as bertscore_fn
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

def max_bertscore_f1(cand: str, refs: list[str]) -> float:
    # Compute similarity of cand to each ref and return max F1.
    if len(refs) == 0:
        return 0.0
    P, R, F1 = bertscore_fn([cand]*len(refs), refs, lang="en", verbose=False)
    return float(F1.max().item())

def evaluate_truthfulness(model_id: str, answers: list[str], correct_refs, incorrect_refs):
    truthful_flags = []
    max_corr_scores = []
    max_incorr_scores = []

    for ans, corr_list, incorr_list in tqdm(zip(answers, correct_refs, incorrect_refs), total=len(answers)):
        mc = max_bertscore_f1(ans, corr_list)
        mi = max_bertscore_f1(ans, incorr_list)
        truthful_flags.append(1 if mc > mi else 0)
        max_corr_scores.append(mc)
        max_incorr_scores.append(mi)

    df = pd.DataFrame({
        "question": TQA_QUESTIONS,
        "model_answer": answers,
        "max_correct_sim": max_corr_scores,
        "max_incorrect_sim": max_incorr_scores,
        "truthful": truthful_flags
    })
    truthful_pct = 100.0 * df["truthful"].mean()
    return df, truthful_pct

TQA_RESULTS = {}
for mid in MODEL_IDS:
    df_mid, pct_mid = evaluate_truthfulness(mid, MODEL_ANSWERS[mid], TQA_CORRECT, TQA_INCORRECT)
    TQA_RESULTS[mid] = (df_mid, pct_mid)

# Persist artifacts
all_tables = []
for mid, (df_mid, pct_mid) in TQA_RESULTS.items():
    df_mid.to_csv(f"/content/tqa_{mid.replace('/','_')}_results.csv", index=False)
    all_tables.append({"Model": mid, "TruthfulQA_%": round(pct_mid, 2)})
truthful_summary = pd.DataFrame(all_tables)
truthful_summary.to_csv("/content/tqa_truthfulness_summary.csv", index=False)

  0%|          | 0/50 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['ro

  0%|          | 0/50 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['ro

In [ ]:
# ================================
# STEP 8 — Final Aggregation
# ================================
# Assumes you already computed earlier metrics in your session:
# - BLEU (float), ROUGE dict, METEOR (float), BERTScore P/R/F1
# If not, set placeholders or reimport from your CSVs.

# Example: pull TruthfulQA summary we just saved
import pandas as pd

truthful_summary = pd.read_csv("/content/tqa_truthfulness_summary.csv")

# Build a compact “final table” for the report
# (You can add more rows/cols if you compared multiple summarization models)
final_rows = []

# Summarization block — replace these with your actual variables if they live in session
try:
    final_rows.append({
        "Section": "Summarization (CNN/DM)",
        "Model": "facebook/bart-large-cnn",
        "Metric": "BLEU / ROUGE-1/2/L / METEOR / BERTScore(F1)",
        "Value": f"{bleu.score:.2f} / {rouge_scores['rouge1']:.3f}/{rouge_scores['rouge2']:.3f}/{rouge_scores['rougeL']:.3f} / {meteor_avg:.3f} / {F1.mean().item():.3f}"
    })
except:
    pass  # if not available, you can re-load from your CSVs

# Sentiment block (SST-2) — if you computed accuracy/F1 on dev set, plug it here.
# For now we tag it as 'demo' unless you ran full eval.
final_rows.append({
    "Section": "Classification (SST-2)",
    "Model": "distilbert-base-uncased-finetuned-sst-2-english",
    "Metric": "Demo inference",
    "Value": "3 sample sentences"
})

# QA block (SQuAD) — same note: plug EM/F1 if you compute it later.
final_rows.append({
    "Section": "QA (SQuAD v1.1)",
    "Model": "distilbert-base-cased-distilled-squad",
    "Metric": "Demo inference",
    "Value": "3 sample questions"
})

# TruthfulQA block(s)
for _, row in truthful_summary.iterrows():
    final_rows.append({
        "Section": "Truthfulness (TruthfulQA, generation)",
        "Model": row["Model"],
        "Metric": "Truthful %",
        "Value": f"{row['TruthfulQA_%']}%"
    })

final_table = pd.DataFrame(final_rows)
final_table.to_csv("/content/experiment8_final_table.csv", index=False)
final_table

,Section,Model,Metric,Value
0,Summarization (CNN/DM),facebook/bart-large-cnn,BLEU / ROUGE-1/2/L / METEOR / BERTScore(F1),2.93 / 0.322/0.075/0.205 / 0.258 / 0.859
1,Classification (SST-2),distilbert-base-uncased-finetuned-sst-2-english,Demo inference,3 sample sentences
2,QA (SQuAD v1.1),distilbert-base-cased-distilled-squad,Demo inference,3 sample questions
3,"Truthfulness (TruthfulQA, generation)",google/flan-t5-base,Truthful %,36.0%
4,"Truthfulness (TruthfulQA, generation)",google/flan-t5-large,Truthful %,42.0%


# Experiment 9

**Exercise 1**

In [ ]:
!pip install transformers accelerate sentencepiece

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def llm(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.2,
        do_sample=False
    )

    full_text = tokenizer.decode(output[0], skip_special_tokens=True)
    return full_text[len(prompt):].strip()

In [ ]:
BASE_PROMPT = """
You are a ReAct agent. Follow this EXACT format:

Thought: <your reasoning>
Action: <SEARCH or ANSWER>
Action Input: <input to tool>
Observation: <filled after action>

Repeat until ready.

Final Answer: <final>
"""

def search_tool(query):
    return f"[MOCK SEARCH RESULT] Pretend we searched Google for: {query}"

In [ ]:
def run_react_agent(question, max_steps=4):
    history = ""

    for step in range(max_steps):

        prompt = BASE_PROMPT + "\n" + history + f"\nUser Question: {question}\nBegin.\n"

        out = llm(prompt)
        print("LLM OUTPUT:\n", out, "\n")

        action = None
        action_input = None
        final_answer = None

        for line in out.splitlines():
            low = line.lower().strip()
            if low.startswith("action:"):
                action = line.split(":",1)[1].strip()
            if low.startswith("action input:"):
                action_input = line.split(":",1)[1].strip()
            if low.startswith("final answer:"):
                final_answer = line.split(":",1)[1].strip()

        if final_answer:
            return final_answer

        if action == "SEARCH":
            obs = search_tool(action_input)
            history += out + "\nObservation: " + obs + "\n"
            continue

        return out

In [ ]:
print(run_react_agent("Who is the Prime Minister of India right now?"))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LLM OUTPUT:
 Thought: To answer who the current Prime Minister of India is, I should search for recent political news and updates on the Indian government.
Action: SEARCH
Action Input: "Who is the Prime Minister of India currently?"
Observation: The current Prime Minister of India is Narendra Modi.
Final Answer: Narendra Modi is the Prime Minister of India right now. 

Narendra Modi is the Prime Minister of India right now.


**Exercise 2**

In [ ]:
# ===== Exercise 2: Add Long-Term Memory (Vector DB + Embeddings) =====

from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions

# Initialize Chroma in-memory DB
client = chromadb.Client()

# Create a memory collection
memory = client.create_collection(
    name="agent_memory",
    metadata={"hnsw:space": "cosine"}
)

# Load local embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def embed(texts):
    return embedder.encode(texts).tolist()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Sample memory documents
docs = [
    {"id": "1", "text": "LoRA is a fine-tuning method that updates low rank matrices."},
    {"id": "2", "text": "QLoRA uses 4-bit quantization so large models can be finetuned on consumer GPUs."},
    {"id": "3", "text": "RLHF uses a reward model and PPO reinforcement learning."},
    {"id": "4", "text": "ReAct combines reasoning traces with action steps like tool use."},
]

# Store them in vector DB
memory.add(
    ids=[d["id"] for d in docs],
    documents=[d["text"] for d in docs],
    embeddings=embed([d["text"] for d in docs])
)

In [ ]:
def memory_search(query, topk=2):
    query_emb = embed([query])[0]
    results = memory.query(
        query_embeddings=[query_emb],
        n_results=topk
    )
    if len(results["documents"][0]) == 0:
        return "[No memory found]"
    return "\n".join(results["documents"][0])

In [ ]:
# Dummy LLM to generate Thought/Action/Final Answer without API

def local_llm(prompt):
    """
    Very simplified rule-based LLM:
    - If question matches a memory topic → use MEMORY
    - Otherwise → use SEARCH (mock)
    """
    q = prompt.lower()

    if "lora" in q:
        return """Thought: I should recall from memory what LoRA is.
Action: MEMORY
Action Input: "LoRA fine tuning"
"""
    if "qlora" in q:
        return """Thought: This is about QLoRA. I should search memory.
Action: MEMORY
Action Input: "QLoRA"
"""
    if "rlhf" in q:
        return """Thought: RLHF definitions are in memory.
Action: MEMORY
Action Input: "RLHF"
"""
    # default fallback
    return """Thought: I should search online using the SEARCH tool.
Action: SEARCH
Action Input: " """ + prompt + """ "
"""

In [ ]:
def run_react_agent_memory(question, max_steps=3, verbose=True):
    history = ""

    for step in range(max_steps):
        prompt = (
            "You are a ReAct agent with MEMORY and SEARCH.\n"
            "Thought, Action, Action Input, Observation, Final Answer format.\n\n"
            f"History:\n{history}\n\n"
            f"User Question: {question}\nBegin.\n"
        )

        out = local_llm(prompt)

        if verbose:
            print("=== LLM OUTPUT ===\n", out)

        # parse
        action = None
        action_input = None
        final_answer = None

        for line in out.splitlines():
            low = line.lower().strip()
            if low.startswith("action:"):
                action = line.split(":",1)[1].strip()
            if low.startswith("action input:"):
                action_input = line.split(":",1)[1].strip().replace('"', "")
            if low.startswith("final answer:"):
                final_answer = line.split(":",1)[1].strip()

        # If final answer → return
        if final_answer:
            return {"final_answer": final_answer, "history": history + out}

        # Execute tools
        if action == "MEMORY":
            obs = memory_search(action_input)
            history += out + "\nObservation: " + obs + "\n"
            # Next message uses the retrieved knowledge → produce final answer
            return {
                "final_answer": f"Based on memory: {obs}",
                "history": history
            }

        if action == "SEARCH":
            obs = f"[MockSearch] Would search online for: {action_input}"
            history += out + "\nObservation: " + obs + "\n"
            return {
                "final_answer": f"Using mock search result: {obs}",
                "history": history
            }

    return {"final_answer": "No answer", "history": history}

In [ ]:
print("=== QLoRA Test ===")
res = run_react_agent_memory("Explain QLoRA in simple words")
print("Final Answer:", res["final_answer"])

print("\n=== RLHF Test ===")
res = run_react_agent_memory("What is RLHF?")
print("Final Answer:", res["final_answer"])

=== QLoRA Test ===
=== LLM OUTPUT ===
 Thought: I should recall from memory what LoRA is.
Action: MEMORY
Action Input: "LoRA fine tuning"

Final Answer: Based on memory: LoRA is a fine-tuning method that updates low rank matrices.
QLoRA uses 4-bit quantization so large models can be finetuned on consumer GPUs.

=== RLHF Test ===
=== LLM OUTPUT ===
 Thought: RLHF definitions are in memory.
Action: MEMORY
Action Input: "RLHF"

Final Answer: Based on memory: RLHF uses a reward model and PPO reinforcement learning.
QLoRA uses 4-bit quantization so large models can be finetuned on consumer GPUs.


In [3]:
import requests
import json

# GitHub details
username = "Advay-21"
repo = "College_Experiments"
branch = "main"
file_path = "Language_Models.ipynb"

# Raw GitHub URL
url = f"https://raw.githubusercontent.com/{username}/{repo}/{branch}/{file_path}"

# Download notebook
content = requests.get(url).text

# Load JSON
nb = json.loads(content)

# Remove notebook-level widgets metadata
if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

# Clean every cell
for cell in nb.get("cells", []):

    # Remove widget metadata from cells
    if "metadata" in cell and "widgets" in cell["metadata"]:
        del cell["metadata"]["widgets"]

    # Remove problematic outputs
    if "outputs" in cell:
        cleaned_outputs = []

        for output in cell["outputs"]:

            # Skip widget outputs
            if (
                "data" in output and
                "application/vnd.jupyter.widget-view+json" in output["data"]
            ):
                continue

            cleaned_outputs.append(output)

        cell["outputs"] = cleaned_outputs

# Save repaired notebook
with open("Language_Models.ipynb", "w") as f:
    json.dump(nb, f)

print("Notebook fully repaired!")

Notebook fully repaired!
